In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkFiles
import pyspark.sql.functions as F

# Requiere winutils.exe en HADOOP_HOME\bin (necesario en Windows para addFile)
os.environ["HADOOP_HOME"] = r"C:\hadoop\winutils\hadoop-3.3.6"
os.environ["hadoop.home.dir"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" + os.environ["PATH"]

# Abrir sesión de Spark (reinicia el kernel antes de ejecutar esta celda para que la JVM arranque con HADOOP_HOME ya definido)
spark = SparkSession.builder.appName("ProcesamientoAvanzado").getOrCreate()

url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/customers-100000.csv"
spark.sparkContext.addFile(url)
df_customers = spark.read.csv(SparkFiles.get("customers-100000.csv"), header=True, inferSchema=True)
df_customers.show()

+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name| Last Name|             Company|             City|             Country|             Phone 1|             Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher|    Hatfieldshire|             Eritrea|  274.188.8773x41185|001-215-760-4642x969|gabriellehartman@...|       2021-11-11|https://www.mccar...|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|   Robertsonburgh|            Botswana|        283-236-9529| (189)129-8356x63741|kstafford@sexton.com|

In [2]:
# Ver tamaño del dataframe (equivalente a df.shape en pandas)
shape = (df_customers.count(), len(df_customers.columns))
print(f"Rows: {shape[0]}, Columns: {shape[1]}")

Rows: 100000, Columns: 12


In [5]:
# Manejo de valores nulos
df_customers.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_customers.columns]).show()

+-----+-----------+----------+---------+-------+----+-------+-------+-------+-----+-----------------+-------+
|Index|Customer Id|First Name|Last Name|Company|City|Country|Phone 1|Phone 2|Email|Subscription Date|Website|
+-----+-----------+----------+---------+-------+----+-------+-------+-------+-----+-----------------+-------+
|    0|          0|         0|        0|      0|   0|      0|      0|      0|    0|                0|      0|
+-----+-----------+----------+---------+-------+----+-------+-------+-------+-----+-----------------+-------+



In [ ]:
# Misma comprobación de nulos, pero escrita paso a paso para que sea más fácil de seguir (didáctico)
total_filas = df_customers.count()

for columna in df_customers.columns:
    # Filtramos las filas donde esta columna concreta es nula y contamos cuántas hay
    columna_es_nula = F.col(columna).isNull()
    filas_nulas = df_customers.filter(columna_es_nula)
    numero_de_nulos = filas_nulas.count()

    print(f"Columna '{columna}': {numero_de_nulos} valores nulos de {total_filas} filas")